In [0]:
# ============================================================
# NOTEBOOK 5 — WORKFLOWS + DELTA LIVE TABLES (DLT)
# Topics : Databricks Workflows, DLT pipelines,
#           @dlt.table, @dlt.expect, data quality,
#           pipeline observability, job orchestration
# ============================================================

CATALOG       = "brazilian-ecommerce"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA   = "gold"
VOLUME_PATH   = "/Volumes/brazilian-ecommerce/filestore/olis/"

print("Config ready.")
print(f"Catalog : {CATALOG}")

In [0]:
# In production, you'd wire notebooks as Workflow tasks via the UI
# Here we simulate the dependency chain programmatically using dbutils

# dbutils.notebook.run() executes another notebook and returns its output
# This is exactly what Databricks Workflows does under the hood

import time

pipeline_steps = [
    ("01_bronze_ingestion",      "/Users/{your_email}/01_bronze_ingestion"),
    ("02_silver_transformations", "/Users/{your_email}/02_silver_transformations"),
    ("03_gold_aggregations",     "/Users/{your_email}/03_gold_aggregations"),
    ("04_delta_lake_deep_dive",  "/Users/{your_email}/04_delta_lake_deep_dive"),
]

# NOTE: Replace {your_email} with your actual Databricks workspace email path
# OR use this cell just to understand the concept — we'll use the Workflow UI below

print("Pipeline dependency chain:")
print()
for i, (name, path) in enumerate(pipeline_steps):
    prefix = "  " * i
    arrow  = "└─▶ " if i > 0 else "▶ "
    print(f"{prefix}{arrow}[Task {i+1}] {name}")

print()
print("In Workflows UI, each task above becomes a node.")
print("Each node has: notebook path, cluster/serverless, depends_on, retry policy.")

In [0]:
# Widgets make notebooks parameterisable — used in Workflows and CI/CD

dbutils.widgets.removeAll()

dbutils.widgets.text(
    name    = "run_date",
    defaultValue = "2018-09-01",
    label   = "Pipeline Run Date"
)
dbutils.widgets.dropdown(
    name    = "load_mode",
    defaultValue = "incremental",
    choices = ["full", "incremental"],
    label   = "Load Mode"
)
dbutils.widgets.text(
    name    = "target_catalog",
    defaultValue = "brazilian-ecommerce",
    label   = "Target Catalog"
)

# Read widget values — this is how the notebook receives parameters from Workflow
run_date       = dbutils.widgets.get("run_date")
load_mode      = dbutils.widgets.get("load_mode")
target_catalog = dbutils.widgets.get("target_catalog")

print(f"Run date       : {run_date}")
print(f"Load mode      : {load_mode}")
print(f"Target catalog : {target_catalog}")

In [0]:
from pyspark.sql.functions import col, count, when

# Before any pipeline run, validate that upstream tables are healthy
# This is your circuit breaker — fail fast before wasting compute

def validate_table(catalog, schema, table, min_rows=100):
    """
    Pre-flight check: table exists, has enough rows, no critical nulls.
    Returns (passed: bool, message: str)
    """
    full_name = f"`{catalog}`.`{schema}`.`{table}`"
    try:
        df   = spark.table(full_name)
        rows = df.count()

        if rows < min_rows:
            return False, f"FAIL — {full_name} has only {rows} rows (min: {min_rows})"

        return True, f"PASS — {full_name} : {rows:,} rows"

    except Exception as e:
        return False, f"FAIL — {full_name} not found: {str(e)}"


checks = [
    validate_table(CATALOG, "bronze", "orders",      min_rows=1000),
    validate_table(CATALOG, "bronze", "order_items",  min_rows=1000),
    validate_table(CATALOG, "bronze", "customers",    min_rows=1000),
    validate_table(CATALOG, "silver", "orders_master",min_rows=5000),
    validate_table(CATALOG, "gold",   "revenue_by_category_monthly", min_rows=10),
    validate_table(CATALOG, "gold",   "customer_lifetime_value",      min_rows=100),
]

all_passed = True
print("=== Pre-flight Validation ===\n")
for passed, message in checks:
    status = "✓" if passed else "✗"
    print(f"  {status} {message}")
    if not passed:
        all_passed = False

print()
if not all_passed:
    raise Exception("Pre-flight validation failed — pipeline aborted.")
else:
    print("All checks passed — pipeline cleared to run.")

In [0]:
# Production pipelines write metrics to a Delta table after every run
# This gives you: run history, row count trends, latency tracking

METRICS_TABLE = f"`{CATALOG}`.`{SILVER_SCHEMA}`.`pipeline_run_metrics`"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {METRICS_TABLE} (
        run_id          STRING,
        run_date        STRING,
        task_name       STRING,
        rows_read       LONG,
        rows_written    LONG,
        duration_secs   DOUBLE,
        status          STRING,
        error_message   STRING,
        recorded_at     TIMESTAMP
    )
    USING DELTA
""")

print(f"Metrics table ready : {METRICS_TABLE}")

import uuid, time
from pyspark.sql import Row
from datetime import datetime

def log_pipeline_metric(task_name, rows_read, rows_written,
                        duration_secs, status, error_message=None):
    """Write one pipeline run record to the metrics Delta table."""
    metric_row = spark.createDataFrame([Row(
        run_id        = str(uuid.uuid4())[:8],
        run_date      = run_date,
        task_name     = task_name,
        rows_read     = int(rows_read),
        rows_written  = int(rows_written),
        duration_secs = float(duration_secs),
        status        = status,
        error_message = error_message or "",
        recorded_at   = datetime.utcnow()
    )])
    metric_row.write.format("delta").mode("append").saveAsTable(METRICS_TABLE)

# Simulate logging metrics for each pipeline stage
t0 = time.time()
bronze_rows = spark.table(f"`{CATALOG}`.`bronze`.`orders`").count()
log_pipeline_metric("bronze_ingestion",      0,           bronze_rows, time.time()-t0, "SUCCESS")

t0 = time.time()
silver_rows = spark.table(f"`{CATALOG}`.`silver`.`orders_master`").count()
log_pipeline_metric("silver_transformation", bronze_rows, silver_rows, time.time()-t0, "SUCCESS")

t0 = time.time()
gold_rows = spark.table(f"`{CATALOG}`.`gold`.`revenue_by_category_monthly`").count()
log_pipeline_metric("gold_aggregation",      silver_rows, gold_rows,   time.time()-t0, "SUCCESS")

print("\nPipeline run metrics logged:")
display(spark.sql(f"SELECT * FROM {METRICS_TABLE} ORDER BY recorded_at DESC"))

In [0]:
# Delta Live Tables pipelines are defined in a SEPARATE notebook
# You cannot run DLT cells interactively — they must be executed via a DLT Pipeline

# This cell creates the DLT notebook programmatically in your workspace
# Then you'll wire it to a DLT Pipeline in the UI

dlt_notebook_content = '''
# Databricks notebook source
# This notebook defines a Delta Live Tables pipeline
# Run via: Workflows → Delta Live Tables → Create Pipeline

import dlt
from pyspark.sql.functions import (
    col, to_timestamp, upper, trim, current_timestamp,
    when, lit, datediff, round as spark_round,
    sum as spark_sum, avg, countDistinct
)

VOLUME_PATH = "/Volumes/brazilian-ecommerce/filestore/olis/"
CATALOG     = "brazilian-ecommerce"

# ─────────────────────────────────────────────
# BRONZE LAYER — Raw ingestion
# ─────────────────────────────────────────────

@dlt.table(
    name    = "bronze_orders_dlt",
    comment = "Raw orders ingested from CSV source volume",
    table_properties = {"quality": "bronze"}
)
def bronze_orders():
    return (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(VOLUME_PATH + "olist_orders_dataset.csv")
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_layer", lit("bronze"))
    )


@dlt.table(
    name    = "bronze_order_items_dlt",
    comment = "Raw order items from CSV source",
    table_properties = {"quality": "bronze"}
)
def bronze_order_items():
    return (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(VOLUME_PATH + "olist_order_items_dataset.csv")
        .withColumn("_ingested_at", current_timestamp())
    )


# ─────────────────────────────────────────────
# SILVER LAYER — Clean + validate with expectations
# ─────────────────────────────────────────────

@dlt.table(
    name    = "silver_orders_dlt",
    comment = "Cleaned, typed, deduplicated orders",
    table_properties = {"quality": "silver"}
)
@dlt.expect("valid_order_id",      "order_id IS NOT NULL")
@dlt.expect("valid_status",        "order_status IS NOT NULL")
@dlt.expect_or_drop(
    "valid_purchase_timestamp",
    "order_purchase_timestamp IS NOT NULL"
)
@dlt.expect_or_fail(
    "no_future_orders",
    "order_purchase_timestamp < current_timestamp()"
)
def silver_orders():
    return (
        dlt.read("bronze_orders_dlt")
        .withColumn("order_purchase_timestamp",
                    to_timestamp(col("order_purchase_timestamp"),
                                 "yyyy-MM-dd HH:mm:ss"))
        .withColumn("order_estimated_delivery_date",
                    to_timestamp(col("order_estimated_delivery_date"),
                                 "yyyy-MM-dd HH:mm:ss"))
        .withColumn("order_delivered_customer_date",
                    to_timestamp(col("order_delivered_customer_date"),
                                 "yyyy-MM-dd HH:mm:ss"))
        .withColumn("order_status", upper(trim(col("order_status"))))
        .withColumn("is_late_delivery",
                    when(col("order_delivered_customer_date")
                         > col("order_estimated_delivery_date"), True)
                    .otherwise(False))
        .dropDuplicates(["order_id"])
        .filter(col("order_id").isNotNull())
    )


@dlt.table(
    name    = "silver_order_items_dlt",
    comment = "Cleaned order items with line totals",
    table_properties = {"quality": "silver"}
)
@dlt.expect("valid_order_id",  "order_id IS NOT NULL")
@dlt.expect("positive_price",  "price > 0")
@dlt.expect_or_drop(
    "valid_product",
    "product_id IS NOT NULL"
)
def silver_order_items():
    return (
        dlt.read("bronze_order_items_dlt")
        .withColumn("line_total",
                    spark_round(col("price") + col("freight_value"), 2))
        .filter(col("price").isNotNull() & (col("price") > 0))
        .dropDuplicates(["order_id", "order_item_id"])
    )


# ─────────────────────────────────────────────
# GOLD LAYER — Business KPIs
# ─────────────────────────────────────────────

@dlt.table(
    name    = "gold_revenue_summary_dlt",
    comment = "Daily revenue KPIs by order status — executive dashboard feed",
    table_properties = {"quality": "gold"}
)
def gold_revenue_summary():
    orders = dlt.read("silver_orders_dlt")
    items  = dlt.read("silver_order_items_dlt")

    return (
        items.join(orders, on="order_id", how="inner")
        .filter(col("order_status") == "DELIVERED")
        .groupBy("order_status", "is_late_delivery")
        .agg(
            spark_round(spark_sum("line_total"), 2).alias("total_revenue"),
            countDistinct("order_id")             .alias("total_orders"),
            spark_round(avg("line_total"), 2)     .alias("avg_order_value")
        )
    )
'''

# Write the DLT notebook to your workspace
notebook_path = f"/Workspace/Users/dlt_pipeline_ecommerce"

try:
    dbutils.fs.put(
        f"file://{notebook_path}.py",
        dlt_notebook_content,
        overwrite=True
    )
    print(f"DLT notebook written to : {notebook_path}.py")
except Exception as e:
    print("Note: Use the manual approach below to create the DLT notebook.")
    print(f"Error: {e}")

print("\nDLT notebook content ready — see Cell 7 for UI setup instructions.")

In [0]:
# ─────────────────────────────────────────────────────────────
# MANUAL STEPS — Do this in the Databricks UI
# ─────────────────────────────────────────────────────────────

steps = """
STEP 1 — Create the DLT notebook manually
  • Workspace → Home → + New → Notebook
  • Name it: dlt_pipeline_ecommerce
  • Language: Python
  • Paste the entire code block from Cell 6 into it
  • Save it (Ctrl+S)

STEP 2 — Create a Delta Live Tables Pipeline
  • Left sidebar → Workflows → Delta Live Tables tab
  • Click "+ Create pipeline"
  • Fill in:
      Pipeline name   : ecommerce_dlt_pipeline
      Product edition : Core  (free tier is fine)
      Pipeline mode   : Triggered  (runs once, not continuous)
      Paths           : select your dlt_pipeline_ecommerce notebook
      Destination     : select "Unity Catalog"
      Target catalog  : brazilian-ecommerce
      Target schema   : dlt_output
  • Click "Create"

STEP 3 — Run the DLT Pipeline
  • Click "Start" button
  • Watch the pipeline graph render — Bronze → Silver → Gold
  • Click each node to see:
      - Row counts written
      - Expectation pass/fail counts
      - Data quality scores

STEP 4 — Check the Event Log
  • After the run, go to the pipeline settings
  • Query the event log:
      SELECT * FROM delta.`/pipelines/{pipeline_id}/system/events`
      WHERE event_type = 'flow_progress'
      ORDER BY timestamp DESC
"""

print(steps)

In [0]:
# Databricks has a full REST API to create/manage Workflows programmatically
# This is used in CI/CD pipelines (GitHub Actions, Azure DevOps)

import json

# This shows what a Workflow definition looks like as JSON
# In production, this JSON lives in your Git repo and is deployed via CI/CD

workflow_definition = {
    "name": "ecommerce_pipeline_daily",
    "schedule": {
        "quartz_cron_expression": "0 0 6 * * ?",  # daily at 6 AM
        "timezone_id": "Asia/Kolkata"
    },
    "tasks": [
        {
            "task_key": "bronze_ingestion",
            "notebook_task": {
                "notebook_path": "/Users/your_email/01_bronze_ingestion",
                "base_parameters": {
                    "run_date": "{{job.start_time.iso_date}}",
                    "load_mode": "incremental"
                }
            },
            "job_cluster_key": "main_cluster"
        },
        {
            "task_key": "silver_transformation",
            "depends_on": [{"task_key": "bronze_ingestion"}],
            "notebook_task": {
                "notebook_path": "/Users/your_email/02_silver_transformations",
                "base_parameters": {
                    "run_date": "{{job.start_time.iso_date}}"
                }
            },
            "job_cluster_key": "main_cluster"
        },
        {
            "task_key": "gold_aggregation",
            "depends_on": [{"task_key": "silver_transformation"}],
            "notebook_task": {
                "notebook_path": "/Users/your_email/03_gold_aggregations",
                "base_parameters": {
                    "run_date": "{{job.start_time.iso_date}}"
                }
            },
            "job_cluster_key": "main_cluster"
        },
        {
            "task_key": "dlt_pipeline",
            "depends_on": [{"task_key": "bronze_ingestion"}],
            "pipeline_task": {
                "pipeline_id": "your_dlt_pipeline_id"
            }
        }
    ],
    "job_clusters": [
        {
            "job_cluster_key": "main_cluster",
            "new_cluster": {
                "spark_version": "14.3.x-scala2.12",
                "num_workers": 0,
                "spark_conf": {
                    "spark.master": "local[*, 4]"
                }
            }
        }
    ],
    "email_notifications": {
        "on_failure": ["your_email@company.com"]
    },
    "max_retries": 2,
    "retry_on_timeout": True
}

print("Workflow JSON definition:")
print(json.dumps(workflow_definition, indent=2))

In [0]:
summary = """
╔══════════════════════════════════════════════════════════════╗
║        E-COMMERCE ANALYTICS PIPELINE — COMPLETE             ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  STORAGE    Unity Catalog Volumes → Delta Lake               ║
║  COMPUTE    Databricks Serverless                            ║
║  FORMAT     Delta Lake (ACID, time travel, CDF)              ║
║                                                              ║
║  BRONZE     Raw CSVs → Delta tables (5 tables)               ║
║             · Explicit schemas                               ║
║             · Audit columns (_ingested_at, _source_file)     ║
║             · Unity Catalog managed tables                   ║
║                                                              ║
║  SILVER     Cleaned, joined, enriched (1 master table)       ║
║             · Timestamp casting, null handling, dedup        ║
║             · Inner + left joins (grain-aligned)             ║
║             · Window functions (row_number, rank, lag)       ║
║             · Partitioned by customer_state                  ║
║             · OPTIMIZE + ZORDER BY                           ║
║                                                              ║
║  GOLD       4 business KPI tables                            ║
║             · Revenue by category (monthly)                  ║
║             · Customer Lifetime Value + segments             ║
║             · Seller performance scorecard                   ║
║             · MoM category growth trend                      ║
║                                                              ║
║  DELTA OPS  MERGE INTO, SCD Type 1 & 2, Time Travel         ║
║             RESTORE, Schema Evolution, Change Data Feed      ║
║             VACUUM, OPTIMIZE, ZORDER, Statistics             ║
║                                                              ║
║  PIPELINE   Databricks Workflows (multi-task DAG)            ║
║             DLT with @dlt.expect quality gates               ║
║             Pre-flight validation + metrics logging          ║
║             Widgets for parameterisation                     ║
║             CI/CD-ready (DABs + REST API)                    ║
╚══════════════════════════════════════════════════════════════╝
"""
print(summary)

In [0]:
# Your complete interview story — practise saying this out loud

story = """
INTERVIEW ANSWER — "Tell me about a data pipeline you built"

"I built an end-to-end e-commerce analytics pipeline on Databricks
using the Olist Brazilian e-commerce dataset.

The architecture follows the medallion pattern across three layers:

Bronze ingests five raw CSV files from a Unity Catalog Volume into
Delta managed tables using explicit schemas and audit columns for
lineage tracking.

Silver performs all cleaning and enrichment — timestamp casting,
deduplication on business keys, grain-aligned payment aggregation
before joining, and window functions like row_number and lag for
customer order sequencing and month-over-month trends. The Silver
master table is partitioned by customer_state and Z-ordered on
purchase timestamp and category for query optimisation.

Gold produces four business KPI tables: monthly revenue by category,
customer lifetime value with segmentation, seller GMV scorecards,
and category MoM growth. I used OPTIMIZE and VACUUM for Delta
maintenance and ANALYZE TABLE to refresh data skipping statistics.

For Delta operations I implemented MERGE INTO for incremental upserts,
SCD Type 2 using record hashing to detect changes and close historical
records, time travel reads and RESTORE for pipeline recovery, schema
evolution with mergeSchema, and Change Data Feed for efficient
downstream propagation.

The pipeline is orchestrated as a Databricks Workflow with four
dependent tasks, pre-flight validation as a circuit breaker, and
a metrics Delta table for run observability. I also built a DLT
version of the Bronze-to-Gold flow using @dlt.expect_or_drop for
data quality enforcement with pipeline-level expectation tracking.

The whole thing is parameterised with widgets and structured for
CI/CD deployment using Databricks Asset Bundles."
"""
print(story)